# Extrapolate end-to-end (intelligrate.extrapolate)

This notebook runs the full `intelligrate.extrapolate` workflow using the installed Python package:
1) nested CV training (OOF outputs)
2) full fit (final model artifact)
3) full predict (new samples + optional evaluation on paired subset)

It expects example input files in `data/HF_sourdough/` relative to the folder where you start Jupyter, and writes outputs to `results/HF_sourdough_pwys/`. You do not need to clone the full repository.


## Install and files
Install Intelligrate into the environment used by this notebook. Core Intelligrate is intended for Python 3.10-3.12 on macOS, Linux, and Windows.

If you already have Jupyter running from the environment you want to use:

```bash
pip install intelligrate
```

If you are creating a new notebook environment, install Jupyter in the same environment:

```bash
pip install intelligrate notebook ipykernel
```

Verify from a terminal:

```bash
python -c "import intelligrate; print('intelligrate import OK')"
intelligrate --help
intelligrate extrapolate --help
intelligrate extrapolate full-predict --help
```

The help output uses placeholders such as `PATH`, `TSV`, or `JOBLIB` to describe values you provide; do not type bracketed usage text such as `[-h]` literally.

Then download this notebook and the GitHub example folder `data/HF_sourdough/` into the same working folder. The expected layout is:

```text
your_working_folder/
  02_extrapolate_train_evaluate_full_fit_predict_HF_sourdough_pwys.ipynb
  data/HF_sourdough/...
  results/                  # created by the notebook
```

The pip package contains the library code. Example notebooks and data are downloaded separately from GitHub.


In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
pwd

## Environment check
This confirms the notebook kernel sees the pip-installed package.


In [ ]:
# Intelligrate should be installed in the active notebook kernel.
# From a terminal before launching Jupyter:
#   pip install intelligrate
# If this environment does not already have Jupyter:
#   pip install notebook ipykernel
# Optional map plots in the subset notebook:
#   pip install "intelligrate[maps]"

# To verify from a terminal:
#   python -c "import intelligrate; print('intelligrate import OK')"
#   intelligrate --help
#   intelligrate extrapolate --help
#   intelligrate extrapolate full-predict --help

from importlib.metadata import version
import intelligrate
print('intelligrate version:', version('intelligrate'))


In [ ]:
# This notebook assumes it is run from a working folder with this structure:
# .
# |-- HF_sourdough_pwys
# |-- data/HF_sourdough/...
# `-- results/                  # created automatically
#
# Download the notebook and the corresponding data/HF_sourdough/ folder from GitHub,
# then start Jupyter from this working folder. The package itself should come from pip.

project_dir = Path.cwd().resolve()
data_root = project_dir / 'data'
data_dir = data_root / 'HF_sourdough'
results_dir = project_dir / 'results' / 'HF_sourdough_pwys'
results_dir.mkdir(parents=True, exist_ok=True)

required_files = ['X_kmers_full.tsv', 'X_kmers.tsv', 'Y_pwys.tsv', 'pwy_to_superclass.tsv']
missing = [str(data_dir / name) for name in required_files if not (data_dir / name).exists()]
if missing:
    raise FileNotFoundError(
        'Missing example input files. Download the matching data folder from GitHub '
        'and keep it next to this notebook. Missing: ' + ', '.join(missing)
    )

print('Working folder:', project_dir)
print('Data folder:', data_dir)
print('Results folder:', results_dir)


## Load inputs
- `X_kmers.tsv`: paired samples (k-mers)
- `X_kmers_full.tsv`: all samples (paired + unpaired, for extrapolation)
- `Y_kos.tsv`: KO profiles for paired samples
- `ko_to_superclass.tsv`: KO -> pathway/superclass mapping, optional


In [ ]:
X_full = pd.read_csv(data_dir / 'X_kmers_full.tsv', sep='	', index_col=0)
X = pd.read_csv(data_dir / 'X_kmers.tsv', sep='	', index_col=0)
Y = pd.read_csv(data_dir / 'Y_pwys.tsv', sep='	', index_col=0)

print('X_full:', X_full.shape)
print('X:', X.shape)
print('Y:', Y.shape)


### Optional: Pre-filter Y before any modeling
If you want to pre-filter KO features (e.g., apply a detection threshold once globally and keep zeroes as informative), do it here **once** and then use `Y` everywhere downstream.
This avoids per-fold KO dropping, but it changes the modeling assumptions.

**Important:** if you do this, pass the filtered `Y` into all downstream calls (training, fixed-param sweep, evaluation, PICRUSt2 comparisons).


In [ ]:
# --- Optional global Y pre-filtering (edit as needed) ---
# Example: apply a detect threshold once and keep zeros as zeros.
# This is global pre-filtering; be aware this uses all samples (no CV isolation).
# If you enable this, replace Y with Y_prefiltered and use it everywhere downstream.
#
# detect_threshold = model_cfg['y_detect_threshold']
# Y_prefiltered = Y.copy()
# Y_prefiltered = Y_prefiltered.mask(Y_prefiltered < detect_threshold, 0.0)
# Y = Y_prefiltered

# NOTE: After this point, make sure any call uses the updated Y:
# - train/_run_once or nested CV inputs
# - fixed_param_oof_knn_on_embedding
# - evaluate_paired_subset
# - PICRUSt2 comparisons (align to the same KO set as Y)


## Parameters
Parameters that we define here correspond to what can be found in the **defaults** as `configs/default.yaml`, but now we expose them here in the notebook for easier editing.


If you want to see every parameter and full explanations, also check:
- Intelligrate extrapolate tutorial on GitHub
- Python API documentation in the installed `intelligrate.extrapolate` package


In [ ]:
# ---- Data file names (used by the training API) ----
data_cfg = {
    'x_full': 'X_kmers_full.tsv',
    'x': 'X_kmers.tsv',
    'y': 'Y_pwys.tsv',
    'picrust2': None, # optional for comparison
    'ko_to_superclass': 'pwy_to_superclass.tsv', # optional for pathway RMSE
}

# ---- Cross-validation settings ----
cv_cfg = {
    # outer_splits/inner_splits: more splits = slower but more stable estimates
    'outer_splits': 5,
    'inner_splits': 3,
    'seed': 0,
    # informed_splits: use metadata to form splits (False by default)
    'informed_splits': False,
}

# ---- X embedding (k-mer -> low-dim) ----
embed_cfg = {
    # min_prev_x_abs: drop k-mers present in fewer samples than this
    'min_prev_x_abs': 10, #30
    # pseudocount_x: added before CLR
    'pseudocount_x': 0.5,
    # n_components: SVD dimensions for the k-mer embedding (X space)
    'n_components': 128, #128
}

# ---- Model / prediction settings ----
model_cfg = {
    # min_prev_y_abs: drop KOs seen in fewer samples than this
    'min_prev_y_abs': 1,
    # y_detect_threshold: detection threshold in TSS space
    'y_detect_threshold': 1000.0,
    # pseudocount_y: added before CLR
    'pseudocount_y': 0.5 / 1e6,

    # KNN hyperparameters (grid for nested CV)
    'neigh_k_grid': [20, 24, 28, 32],
    'tau_mult_grid': [0.5, 1.0, 2.0, 4.0],
    'lam_grid': [0.0],
    'y_latent_k_grid': [0, 10, 20],

    # Metric learning
    'use_metric_learning': True,
    'metric_max_pairs': 5000,
    'metric_ridge_grid': [1.0, 2.5, 5.0],

    # Out-of-distribution shrinkage
    'ood_shrink': True,
    'ood_shrink_inner': True,
    'ood_lam_base': 0.7,
    'ood_lam_cap': 0.5,
    'ood_tau_inflate': False,
}

# ---- Objective weights ----
objective_cfg = {
    # w_dm is the primary objective (Aitchison DM Spearman)
    'w_dm': 1.0,
    'w_wclr': 0.0,
    'w_pw_rmse': 0.0,
    'w_softf1': 0.0,
    'w_jsd': 0.0,
}

# ---- Precision/recall settings ----
prf_cfg = {
    'prf_thresh': 1.0e-6,
    'prf_weight': 'binary', 
}

# ---- Optional metrics ----
metrics_cfg = {
    'compute_wclr': True,
    'compute_jsd': True,
    'compute_pathway_rmse': True,
    'pathway_rmse_per_group': True,
    'pathway_rmse_log1p': True,
}

# ---- Score settings (for the optional global OOF DM check) ----
score_cfg = {
    'min_prev_y_abs': 1,
    'y_detect_threshold': 3000.0,
    'pseudocount_y': 0.5 / 1e6,
}


## 1) Train on paired-samples (nested CV)
We call the training API directly and then write the outputs to `results/`.

This step is to get out-of-fold (OOF) predictions for the paired samples, which can be used for initial model evaluation and calibration/hyperparameter tuning and selection.


In [ ]:
from intelligrate.extrapolate import train as train_model

cfg = {
    'data': data_cfg,
    'cv': cv_cfg,
    'embed': embed_cfg,
    'model': model_cfg,
    'objective': objective_cfg,
    'prf': prf_cfg,
    'metrics': metrics_cfg,
    'score': score_cfg,
}

payload = train_model._run_once(cfg, data_dir=data_dir, out_dir=results_dir)

# Write outputs (same files as the CLI)
oof_clr = payload['oof_clr']
oof_tss = payload['oof_tss']
folds = payload['folds']
run = payload['run']

# Save core outputs
(oof_clr).to_csv(results_dir / 'oof_clr.tsv', sep='	')
(oof_tss).to_csv(results_dir / 'oof_tss.tsv', sep='	')
folds.to_csv(results_dir / 'folds.tsv', sep='	', index=False)
(results_dir / 'summary.json').write_text(json.dumps(run, indent=2))

summary_flat = {k: v for k, v in run.items() if k != 'config'}
pd.DataFrame([summary_flat]).to_csv(results_dir / 'summary.tsv', sep='	', index=False)

print('OBJECTIVE_DM_SPEARMAN_MEAN:', run['objective_dm_spearman_mean'])
print('MODEL_DM_UNION_STRICT:', run['model_dm_union_strict'])


In [ ]:
summary = json.loads((results_dir / 'summary.json').read_text())
summary_keys = [
    'objective_dm_spearman_mean',
    'model_dm_union_strict',
    'picrust2_dm_union_strict',
    'delta_union',
    'runtime_sec',
]

pd.DataFrame([{k: summary.get(k) for k in summary_keys}])


  - objective_dm_spearman_mean: primary training objective, averaged Spearman correlation between true vs. predicted
  Aitchison distance matrices across CV folds (higher = better recovery of sample–sample structure).
  - model_dm_union_strict: KO‑union Spearman score on OOF predictions (union of KOs in truth and prediction, computed in CLR/
  Aitchison space; higher = better overall agreement).
  - picrust2_dm_union_strict: same KO‑union score for PICRUSt2 (baseline), optional (only if PICRUSt2 predictions are provided).
  - delta_union: model_dm_union_strict − picrust2_dm_union_strict (positive means the model beats PICRUSt2 in terms of matrix spearman correlations on the KO union).
  - runtime_sec: total training runtime in seconds.

In [ ]:
folds = pd.read_csv(results_dir / 'folds.tsv', sep='	')
folds.head()

-->> this contains all the 'best' parameter selections for paired sample predictions, which can be considered for evaluation and calibration/hyperparameter tuning and selection.


In [ ]:
#how many rows in oof_tss contain any nans?:
oof_tss.isna().any(axis=1).sum()

### Optional: fixed-parameter sweep for stable hyperparameters

Use this to find a **single fixed hyperparameter set** that performs well for leakage-free OOF.
It evaluates each combo with fixed-parameter OOF and ranks by `dm_union_strict`.

**Important:** if a parameter is *not* listed in `fixed_param_sweep`, the sweep will use the value
from config. For parameters with a `*_grid` (e.g., `neigh_k_grid`), it will use that grid list.
To force a single value, list it explicitly in `fixed_param_sweep`.


for (otpional) pathway evaluation later on, we import also the KO -> pathway mapping:

In [ ]:
#make a dictionary out of ko to superclass mapping:
ko_to_superclass_path = data_dir / 'pwy_to_superclass.tsv'
ko_to_superclass_df = pd.read_csv(ko_to_superclass_path, sep='	', index_col=0)
ko_to_superclass = ko_to_superclass_df['superclass'].to_dict()
ko_to_superclass

In [ ]:
embed_cfg

In [ ]:
from intelligrate.extrapolate.embedding import fit_x_embedding_svd_clr
from intelligrate.extrapolate.fixed_param_sweep import run_fixed_param_sweep_explicit

# Fit embedding on all samples (X_full)
embed = fit_x_embedding_svd_clr(
    X_full,
    min_prev_x_abs=int(embed_cfg['min_prev_x_abs']),
    pseudocount_x=float(embed_cfg['pseudocount_x']),
    n_components=int(embed_cfg['n_components']),
    seed=int(cv_cfg['seed']),
)

embed_path = results_dir / 'embed.joblib'
joblib.dump(embed, embed_path)
print('Saved embedding to:', embed_path)

# Define a small fixed-parameter sweep (uses the same X_full/X/Y/ko_to_superclass as above)
cfg["fixed_param_sweep"] = {
    "neigh_k": [24, 28, 32],
    "tau_mult": [0.5, 1.0],
    "y_latent_k": [10, 20],
    "metric_ridge": [1.0, 2.5],
}

#order X and Y by same index:
X = X.loc[Y.index]


sweep_out = results_dir / "fixed_param_sweep.tsv"
sweep_df = run_fixed_param_sweep_explicit(
    X_full=X_full,
    X=X,
    Y=Y,
    ko_to_superclass=ko_to_superclass,
    out_path=sweep_out,
    cv_cfg=cv_cfg,
    embed_cfg=embed_cfg,
    model_cfg=model_cfg,
    prf_cfg=prf_cfg,
    metrics_cfg=metrics_cfg,
    sweep_cfg=cfg["fixed_param_sweep"],
    embed=embed,
)
sweep_df.head()


In [ ]:
#check which sweep params give the best dm_union_strict:
sweep_df.sort_values('dm_union_strict', ascending=False).head()

-> in this case the best config is neight_k = 28, tau_mult = 0.5, y_latent_k = 10, metric_ridge = 1.0

### now if you want to try a few more fixed parameter combos, go ahead, otherwise skip and continue with the full fit of the final model further down

In [ ]:
# Define a small fixed-parameter sweep (uses the same X_full/X/Y/ko_to_superclass as above), round 2:
cfg["fixed_param_sweep"] = {
    "neigh_k": [27, 28, 29],
    "tau_mult": [0.25, 0.35, 0.5, 0.65, 0.8],
    "y_latent_k": [15, 120, 25, 30],
    "metric_ridge": [1.0],
}

sweep_out2 = results_dir / "fixed_param_sweep2.tsv"
sweep_df2 = run_fixed_param_sweep_explicit(
    X_full=X_full,
    X=X,
    Y=Y,
    ko_to_superclass=ko_to_superclass,
    out_path=sweep_out,
    cv_cfg=cv_cfg,
    embed_cfg=embed_cfg,
    model_cfg=model_cfg,
    prf_cfg=prf_cfg,
    metrics_cfg=metrics_cfg,
    sweep_cfg=cfg["fixed_param_sweep"],
    embed=embed,
)
sweep_df2.head()

In [ ]:
sweep_df2.sort_values('dm_union_strict', ascending=False).head()

-> higher neigh_k seems to improve performance slightly! tau is stable at 0.35. dm_union_raw increased 3%!

In [ ]:
# Define a small fixed-parameter sweep (uses the same X_full/X/Y/ko_to_superclass as above), round 3: keep best ones, sweep y_latent_k
cfg["fixed_param_sweep"] = {
    "neigh_k": [28],
    "tau_mult": [0.3, 0.35, 0.4],
    "y_latent_k": [12, 13, 14, 15, 16, 17, 18, 19, 20],
    "metric_ridge": [1.0],
}

sweep_out3 = results_dir / "fixed_param_sweep3.tsv"
sweep_df3 = run_fixed_param_sweep_explicit(
    X_full=X_full,
    X=X,
    Y=Y,
    ko_to_superclass=ko_to_superclass,
    out_path=sweep_out,
    cv_cfg=cv_cfg,
    embed_cfg=embed_cfg,
    model_cfg=model_cfg,
    prf_cfg=prf_cfg,
    metrics_cfg=metrics_cfg,
    sweep_cfg=cfg["fixed_param_sweep"],
    embed=embed,
)
sweep_df3.head()

In [ ]:
sweep_df3.sort_values('dm_union_strict', ascending=False).head()

-> minimal, keep y_latent_k = 10

In [ ]:
# Define a small fixed-parameter sweep (uses the same X_full/X/Y/ko_to_superclass as above), round 4: keep best one, sweep y_detect_threshold
cfg["fixed_param_sweep"] = {
    "neigh_k": [28],
    "tau_mult": [0.4],
    "y_latent_k": [10, 11, 12, 13],
    "metric_ridge": [1.0],
    "y_detect_threshold": [0, 100, 1000.0, 1500.0, 2000.0],
    "metric_max_pairs": [3000, 5000, 7000],
}

sweep_out4 = results_dir / "fixed_param_sweep4.tsv"
sweep_df4 = run_fixed_param_sweep_explicit(
    X_full=X_full,
    X=X,
    Y=Y,
    ko_to_superclass=ko_to_superclass,
    out_path=sweep_out,
    cv_cfg=cv_cfg,
    embed_cfg=embed_cfg,
    model_cfg=model_cfg,
    prf_cfg=prf_cfg,
    metrics_cfg=metrics_cfg,
    sweep_cfg=cfg["fixed_param_sweep"],
    embed=embed,
)
sweep_df4.head()

In [ ]:
sweep_df4.sort_values('dm_union_strict', ascending=False).head()

In [ ]:
# Define a small fixed-parameter sweep (uses the same X_full/X/Y/ko_to_superclass as above), round 4: keep best one, sweep y_detect_threshold
cfg["fixed_param_sweep"] = {
    "neigh_k": [28],
    "tau_mult": [0.4],
    "y_latent_k": [12],
    "metric_ridge": [1.0],
    "min_prev_y_abs": [0, 1, 2, 3, 5, 10],
    "y_detect_threshold": [0, 500, 750, 1000.0, 1250, 1500.0],
    "metric_max_pairs": [3000],
}

sweep_out5 = results_dir / "fixed_param_sweep5.tsv"
sweep_df5 = run_fixed_param_sweep_explicit(
    X_full=X_full,
    X=X,
    Y=Y,
    ko_to_superclass=ko_to_superclass,
    out_path=sweep_out,
    cv_cfg=cv_cfg,
    embed_cfg=embed_cfg,
    model_cfg=model_cfg,
    prf_cfg=prf_cfg,
    metrics_cfg=metrics_cfg,
    sweep_cfg=cfg["fixed_param_sweep"],
    embed=embed,
)
sweep_df5.head()

In [ ]:
sweep_df5.sort_values('dm_union_strict', ascending=False).head()

In [ ]:
# Define a small fixed-parameter sweep (uses the same X_full/X/Y/ko_to_superclass as above), round 4: keep best one, sweep y_detect_threshold
cfg["fixed_param_sweep"] = {
    "neigh_k": [28],
    "tau_mult": [0.4],
    "y_latent_k": [12],
    "metric_ridge": [1.0],
    "min_prev_y_abs": [0],
    "y_detect_threshold": [1250.0],
    "metric_max_pairs": [3000],
}

sweep_out6 = results_dir / "fixed_param_sweep6.tsv"
sweep_df6 = run_fixed_param_sweep_explicit(
    X_full=X_full,
    X=X,
    Y=Y,
    ko_to_superclass=ko_to_superclass,
    out_path=sweep_out,
    cv_cfg=cv_cfg,
    embed_cfg=embed_cfg,
    model_cfg=model_cfg,
    prf_cfg=prf_cfg,
    metrics_cfg=metrics_cfg,
    sweep_cfg=cfg["fixed_param_sweep"],
    embed=embed,
)
sweep_df6.head()

In [ ]:
sweep_df6.sort_values('dm_union_strict', ascending=False).head()

-> that's the final choice for now (sweep_df6)!

## 2) Full fit (final model)
We fit the embedding on all k-mer samples (same as before, but now parameters are exposed in case we would still want to update/improve some based on previous grid scans), then fit a final model on paired samples.

To pick hyperparameters, we take the most common (mode) values from the CV folds.
You can also set them manually.


### option 1 to get the best params from fits without sweep:

In [ ]:

from intelligrate.extrapolate.full_fit import fit_final_model, save_model



# Choose final hyperparameters from folds (mode) -> the ones that were most often selected
def mode_or_first(series, default):
    if series is None or series.empty:
        return default
    return series.mode().iloc[0]

neigh_k = int(mode_or_first(folds.get('neigh_k'), model_cfg['neigh_k_grid'][0]))
tau_mult = float(mode_or_first(folds.get('tau_mult'), model_cfg['tau_mult_grid'][0]))
lam = float(mode_or_first(folds.get('lam'), model_cfg['lam_grid'][0]))
y_latent_k = int(mode_or_first(folds.get('y_latent_k'), model_cfg['y_latent_k_grid'][0]))
metric_ridge = float(mode_or_first(folds.get('metric_ridge'), model_cfg['metric_ridge_grid'][0]))

print('Selected hyperparameters:', neigh_k, tau_mult, lam, y_latent_k, metric_ridge)

### option 2 to get the best params from fixed param sweep (which we will do now!)

In [ ]:
from intelligrate.extrapolate.full_fit import fit_final_model, save_model

#which sweep_df should be considered? adapt here
sweep_final = sweep_df6.copy()

# Choose final hyperparameters from folds (mode) -> the ones that were most often selected
def mode_or_first(series, default):
    if series is None or series.empty:
        return default
    return series.mode().iloc[0]

neigh_k = int(mode_or_first(sweep_final.get('neigh_k'), model_cfg['neigh_k_grid'][0]))
tau_mult = float(mode_or_first(sweep_final.get('tau_mult'), model_cfg['tau_mult_grid'][0]))
lam = float(mode_or_first(sweep_final.get('lam'), model_cfg['lam_grid'][0]))
y_latent_k = int(mode_or_first(sweep_final.get('y_latent_k'), model_cfg['y_latent_k_grid'][0]))
metric_ridge = float(mode_or_first(sweep_final.get('metric_ridge'), model_cfg['metric_ridge_grid'][0]))
min_prev_y_abs = int(mode_or_first(sweep_final.get('min_prev_y_abs'), model_cfg['min_prev_y_abs']))
y_detect_threshold = float(mode_or_first(sweep_final.get('y_detect_threshold'), model_cfg['y_detect_threshold']))
metric_max_pairs = int(mode_or_first(sweep_final.get('metric_max_pairs'), model_cfg['metric_max_pairs']))

print('Selected hyperparameters:', neigh_k, tau_mult, lam, y_latent_k, metric_ridge, min_prev_y_abs, y_detect_threshold, metric_max_pairs)

then, we fit the final model which we will then apply for extrapolation.

In [ ]:
model = fit_final_model(
    X_train=X,
    Y_train_tpm=Y,
    embed=embed,
    min_prev_y_abs=min_prev_y_abs,
    y_detect_threshold=y_detect_threshold,
    pseudocount_y=float(model_cfg['pseudocount_y']),
    neigh_k=neigh_k,
    tau_mult=tau_mult,
    lam=lam,
    y_latent_k=y_latent_k,
    use_metric_learning=bool(model_cfg['use_metric_learning']),
    metric_ridge=metric_ridge,
    metric_max_pairs=metric_max_pairs,
    tau_scale_k_nn=int(model_cfg.get('tau_scale_k_nn', 10)),
    ood_shrink=bool(model_cfg.get('ood_shrink', False)),
    ood_lam_base=float(model_cfg.get('ood_lam_base', 0.1)),
    ood_lam_cap=float(model_cfg.get('ood_lam_cap', 0.8)),
    seed=int(cv_cfg['seed']),
)

model_path = save_model(model, results_dir / 'model.joblib')
print('Saved model to:', model_path)

## 3) Full predict (= extrapolate)
We predict for all samples, then (optionally) evaluate on the paired subset if truth is available.


In [ ]:
from intelligrate.extrapolate.full_predict import predict_final_model, evaluate_paired_subset

# Predict for all samples (deployment)
Yhat_clr, Yhat_tss, diag = predict_final_model(X_full, model)

# We will save only the leakage-free version after replacing paired rows.


## Leakage-free paired predictions (fixed-parameter OOF)
We re-predict paired samples using fixed hyperparameters, training only on other samples.
Set outer_splits = N to emulate leave-one-out.


In [ ]:
from intelligrate.extrapolate.cv_knn import fixed_param_oof_knn_on_embedding

oof_fixed_clr, oof_fixed_tss, oof_fixed_folds = fixed_param_oof_knn_on_embedding(
    X=X,
    Y_tpm=Y,
    embed=embed,
    ko_to_superclass=ko_to_superclass,
    outer_splits=5,  # set to len(X) for leave-one-out
    seed=int(cv_cfg['seed']),
    min_prev_y_abs=min_prev_y_abs,
    y_detect_threshold=y_detect_threshold,
    pseudocount_y=float(model_cfg['pseudocount_y']),
    neigh_k=int(neigh_k),
    tau_mult=float(tau_mult),
    lam=float(lam),
    y_latent_k=int(y_latent_k),
    use_metric_learning=bool(model_cfg['use_metric_learning']),
    metric_max_pairs=metric_max_pairs,
    metric_ridge=float(metric_ridge),
    tau_scale_k_nn=int(model_cfg.get('tau_scale_k_nn', 10)),
    ood_shrink=bool(model_cfg.get('ood_shrink', False)),
    ood_lam_base=float(model_cfg.get('ood_lam_base', 0.1)),
    ood_lam_cap=float(model_cfg.get('ood_lam_cap', 0.8)),
    ood_tau_inflate=bool(model_cfg.get('ood_tau_inflate', False)),
    ood_tau_gamma=float(model_cfg.get('ood_tau_gamma', 1.0)),
    informed_splits=bool(cv_cfg.get('informed_splits', False)),
    informed_kmeans_on='X',
    prf_thresh=float(prf_cfg['prf_thresh']),
    prf_weight=str(prf_cfg['prf_weight']),
)

#drop columns if they are all nan or all 0:
oof_fixed_clr = oof_fixed_clr.loc[:, (oof_fixed_clr.notna().any()) & (oof_fixed_clr.sum(axis=0) != 0)]
oof_fixed_tss = oof_fixed_tss.loc[:, (oof_fixed_tss.notna().any()) & (oof_fixed_tss.sum(axis=0) != 0)]


oof_fixed_clr.to_csv(results_dir / 'oof_fixed_clr.tsv', sep='	')
oof_fixed_tss.to_csv(results_dir / 'oof_fixed_tss.tsv', sep='	')
oof_fixed_folds.to_csv(results_dir / 'oof_fixed_folds.tsv', sep='	', index=False)

# Replace paired rows in full predictions with leakage-free OOF predictions
# pred_full.* uses full-fit predictions for unpaired samples, but paired rows are OOF (leakage-free).
Yhat_clr_full_oof = Yhat_clr.copy()
Yhat_tss_full_oof = Yhat_tss.copy()
# Yhat_clr_full_oof.loc[X.index] = oof_fixed_clr.loc[X.index]
# Yhat_tss_full_oof.loc[X.index] = oof_fixed_tss.loc[X.index]
 # Align columns to full prediction columns
oof_fixed_clr_aligned = oof_fixed_clr.reindex(columns=Yhat_clr_full_oof.columns)
oof_fixed_tss_aligned = oof_fixed_tss.reindex(columns=Yhat_tss_full_oof.columns)

if oof_fixed_clr_aligned.isna().any().sum() > 0:
  oof_fixed_clr_aligned = oof_fixed_clr_aligned.fillna(0.0)
if oof_fixed_tss_aligned.isna().any().sum() > 0:
    oof_fixed_tss_aligned = oof_fixed_tss_aligned.fillna(0.0)

Yhat_clr_full_oof.loc[X.index] = oof_fixed_clr_aligned.loc[X.index]
Yhat_tss_full_oof.loc[X.index] = oof_fixed_tss_aligned.loc[X.index]

pred_full_prefix = results_dir / 'pred_full'
Yhat_clr_full_oof.to_csv(pred_full_prefix.with_suffix('.clr.tsv'), sep='	')
Yhat_tss_full_oof.to_csv(pred_full_prefix.with_suffix('.tss.tsv'), sep='	')

# Evaluate leakage-free paired predictions
metrics_oof = evaluate_paired_subset(
    truth_tpm=Y,
    pred_tss=oof_fixed_tss,
    pseudocount=float(model_cfg['pseudocount_y']),
    detect_threshold=y_detect_threshold,
    prf_thresh=float(prf_cfg['prf_thresh']),
    prf_weight=str(prf_cfg['prf_weight']),
    compute_wclr=bool(metrics_cfg['compute_wclr']),
    compute_jsd=bool(metrics_cfg['compute_jsd']),
    compute_pathway=bool(metrics_cfg['compute_pathway_rmse']),
    compute_per_pathway=bool(metrics_cfg['pathway_rmse_per_group']),
    ko_to_group=ko_to_superclass,
    log1p_pathway=bool(metrics_cfg['pathway_rmse_log1p']),
    compute_null=True,
    null_n=50, #decrease for speed, increase for stability
    null_seed=0,
    null_mean_noise_scale=0.0001,
)

pd.DataFrame([metrics_oof]).to_csv(results_dir / 'pred_paired_oof.metrics.tsv', sep='	', index=False)
metrics_oof


### KO predictability score (per-KO confidence)
This combines conditional neighbor dispersion (lower is better) with per‑KO OOF Spearman (higher is better).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from intelligrate.extrapolate.transforms import tss_rows, clr_rows
from intelligrate.extrapolate.embedding import transform_x_embedding_svd_clr
from intelligrate.extrapolate.metrics import ko_confidence_from_oof

# Build CLR truth and predictions
Y_clr = clr_rows(tss_rows(Y), pseudocount=float(model_cfg["pseudocount_y"]))
oof_pred_clr = oof_fixed_clr.reindex(columns=Y_clr.columns)
assert oof_pred_clr.index.equals(Y_clr.index)

#define correlation threshold:
threshold_r0 = 0.30

# Compute KO confidence (stable across datasets)
Z = transform_x_embedding_svd_clr(X, embed)
S = ko_confidence_from_oof(
    Y_true_clr=Y_clr,
    Y_pred_clr=oof_pred_clr,
    Z=Z,
    r0=threshold_r0, # correlation threshold for confidence
    k_nn=5, # neighbors for stability
    R=50, # null replicates for stability, meaning how many random sets are generated to compare to the observed ones for the stability score
    seed=0,
    min_n=5, # min samples for spearman-> min samples with non-nan values for a KO
)

# Top KOs by confidence
S.sort_values("confidence", ascending=False).head(10)

#save to tsv:
S.to_csv(results_dir / 'per_ko_confidence.tsv', sep='	')


In [ ]:
#is there a row with nan in oof_spearman of S?
S['oof_spearman'].isna().sum()

#drop rows with nan in oof_spearman:
S = S.dropna(subset=['oof_spearman'])
S

In [ ]:
#plot the distribution of oof_corr and cond_disp:
plt.figure(figsize=(16,4))
plt.subplot(1,4,1)
plt.hist(S["oof_spearman"].dropna(), bins=50, color='#468892FF', edgecolor=None, alpha=0.7)
plt.xlabel("Correlation", fontsize=14)
plt.ylabel("Frequency", fontsize=14)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.title("OOF correlation per PWY", fontsize=16)
#add the vertical lines for correlation threshold:
plt.axvline(x=threshold_r0, color='black', linestyle='--', linewidth=1)

#also add text for those lines:
plt.text(-0.2, 15, f"Threshold = {threshold_r0}", color='black', fontsize=14)
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

plt.subplot(1,4,2)
plt.hist(S["conf_corr"].dropna(), bins=50, color='#468892FF', edgecolor=None, alpha=0.7)
plt.xlabel("Confidence", fontsize=14)
plt.ylabel("Frequency", fontsize=14)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.title("OOF correlation confidence", fontsize=16)

plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.subplot(1,4,3)
plt.hist(S["conf_stab"].dropna(), bins=50, color='#468892FF', edgecolor=None, alpha=0.7)
plt.xlabel("Confidence", fontsize=14)
plt.ylabel("Frequency", fontsize=14)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.title("OOF dispersion stability confidence", fontsize=16)
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

plt.subplot(1,4,4)
plt.hist(S["confidence"].dropna(), bins=50, color='#468892FF', edgecolor=None, alpha=0.7)
plt.xlabel("Confidence", fontsize=14)
plt.ylabel("Frequency", fontsize=14)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.title("OOF confidence per PWY", fontsize=16)
#calculate how many KOs have confidence > 0.5:
num_high_confidence = (S["confidence"] > 0.5).sum()
#also calculate the percentage:
percentage_high_confidence = (num_high_confidence / len(S)) * 100
#add the vertical lines for disp_low and disp_high:
plt.axvline(x=0.5, color='black', linestyle='--', linewidth=1)
#also add text for those lines:

plt.text(0.2, 15, f"PWYs with conf > 0.5:\n {num_high_confidence} ({percentage_high_confidence:.1f}%)", color='black', fontsize=14)
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
#save the figure as pdf:
plt.tight_layout()
figure_path = results_dir / 'ko_confidence_distribution.pdf'
plt.savefig(figure_path, format='pdf', bbox_inches='tight')
plt.show()

## Compare against PICRUSt2 + null distributions (are kNN predictions better than predicting the average KO profile of the dataset?)
We compute the same metrics for PICRUSt2 and null distributions, and compare the results.


this will take a while, you can decrease null_n for speed!

In [ ]:
# Load PICRUSt2 predictions if available and compute metrics
picrust_metrics = None
picrust = None
picrust_path = data_dir / data_cfg["picrust2"]
if picrust_path.exists():
    picrust = pd.read_csv(picrust_path, sep="	", index_col=0)
    picrust_metrics = evaluate_paired_subset(
        truth_tpm=Y,
        pred_tss=picrust,
        pseudocount=float(model_cfg["pseudocount_y"]),
        detect_threshold=y_detect_threshold,
        prf_thresh=float(prf_cfg["prf_thresh"]),
        prf_weight=str(prf_cfg["prf_weight"]),
        compute_wclr=bool(metrics_cfg["compute_wclr"]),
        compute_jsd=bool(metrics_cfg["compute_jsd"]),
        compute_pathway=bool(metrics_cfg["compute_pathway_rmse"]),
        compute_per_pathway=bool(metrics_cfg["pathway_rmse_per_group"]),
        ko_to_group=ko_to_superclass,
        log1p_pathway=bool(metrics_cfg["pathway_rmse_log1p"]),
        compute_null=True,
        null_n=50,
        null_seed=0,
        null_mean_noise_scale=0.0001, # null mean noise scale, meaning how much noise is added to the null predictions, the unit is in fraction of the mean of the true values
    )


-> we don't have any picrust for that

now, we aggregate all results together for easier comparison:

In [ ]:
# Build a comparison table: intersection vs union_raw vs union_strict + null baselines
metrics_spec = [
    {"name": "dm_spearman", "intersection": "intersection_dm_spearman", "union_raw": "dm_union_raw", "union_strict": "dm_union_strict",
     "null_sample": "null_sample_dm_spearman_union_strict_mean", "null_feature": "null_feature_dm_spearman_union_strict_mean", "null_mean": "null_mean_dm_spearman_union_strict_mean"},
    {"name": "bray_spearman", "intersection": "intersection_bray_spearman", "union_raw": "bray_union_raw", "union_strict": "bray_union_strict",
     "null_sample": "null_sample_bray_spearman_union_strict_mean", "null_feature": "null_feature_bray_spearman_union_strict_mean", "null_mean": "null_mean_bray_spearman_union_strict_mean"},
    {"name": "procrustes_aitchison", "intersection": "intersection_procrustes_aitchison", "union_raw": "procrustes_aitchison_raw", "union_strict": "procrustes_aitchison_strict",
     "null_sample": "null_sample_procrustes_aitchison_union_strict_mean", "null_feature": "null_feature_procrustes_aitchison_union_strict_mean", "null_mean": "null_mean_procrustes_aitchison_union_strict_mean"},
    {"name": "procrustes_bray", "intersection": "intersection_procrustes_bray", "union_raw": "procrustes_bray_raw", "union_strict": "procrustes_bray_strict",
     "null_sample": "null_sample_procrustes_bray_union_strict_mean", "null_feature": "null_feature_procrustes_bray_union_strict_mean", "null_mean": "null_mean_procrustes_bray_union_strict_mean"},
    {"name": "sample_spearman_tss", "intersection": "sample_spearman_tss_intersection", "union_raw": "sample_spearman_tss_union_raw", "union_strict": "sample_spearman_tss_union_strict",
     "null_sample": "null_sample_sample_spearman_tss_union_strict_mean", "null_feature": "null_feature_sample_spearman_tss_union_strict_mean", "null_mean": "null_mean_sample_spearman_tss_union_strict_mean"},
    {"name": "sample_spearman_clr", "intersection": "sample_spearman_clr_intersection", "union_raw": "sample_spearman_clr_union_raw", "union_strict": "sample_spearman_clr_union_strict",
     "null_sample": "null_sample_sample_spearman_clr_union_strict_mean", "null_feature": "null_feature_sample_spearman_clr_union_strict_mean", "null_mean": "null_mean_sample_spearman_clr_union_strict_mean"},
    {"name": "soft_precision", "intersection": "intersection_soft_precision", "union_raw": "soft_precision_union_raw", "union_strict": "soft_precision",
     "null_sample": "null_sample_soft_precision_union_strict_mean", "null_feature": "null_feature_soft_precision_union_strict_mean", "null_mean": "null_mean_soft_precision_union_strict_mean"},
    {"name": "soft_recall", "intersection": "intersection_soft_recall", "union_raw": "soft_recall_union_raw", "union_strict": "soft_recall",
     "null_sample": "null_sample_soft_recall_union_strict_mean", "null_feature": "null_feature_soft_recall_union_strict_mean", "null_mean": "null_mean_soft_recall_union_strict_mean"},
    {"name": "soft_f1", "intersection": "intersection_soft_f1", "union_raw": "soft_f1_union_raw", "union_strict": "soft_f1",
     "null_sample": "null_sample_soft_f1_union_strict_mean", "null_feature": "null_feature_soft_f1_union_strict_mean", "null_mean": "null_mean_soft_f1_union_strict_mean"},
    {"name": "wclr_mse", "intersection": "intersection_wclr_mse", "union_raw": "wclr_mse_union_raw", "union_strict": "wclr_mse",
     "null_sample": "null_sample_wclr_mse_union_strict_mean", "null_feature": "null_feature_wclr_mse_union_strict_mean", "null_mean": "null_mean_wclr_mse_union_strict_mean"},
    {"name": "jsd", "intersection": "intersection_jsd", "union_raw": "jsd_union_raw", "union_strict": "jsd",
     "null_sample": "null_sample_jsd_union_strict_mean", "null_feature": "null_feature_jsd_union_strict_mean", "null_mean": "null_mean_jsd_union_strict_mean"},
    {"name": "pathway_rmse", "intersection": "intersection_pathway_rmse", "union_raw": "pathway_rmse_union_raw", "union_strict": "pathway_rmse",
     "null_sample": "null_sample_pathway_rmse_union_strict_mean", "null_feature": "null_feature_pathway_rmse_union_strict_mean", "null_mean": "null_mean_pathway_rmse_union_strict_mean"},
]


def _get(d, key):
    if d is None or key is None:
        return None
    return d.get(key)

rows = []
for spec in metrics_spec:
    name = spec["name"]
    model_inter = _get(metrics_oof, spec["intersection"])
    model_raw = _get(metrics_oof, spec["union_raw"])
    model_strict = _get(metrics_oof, spec["union_strict"])
    model_null_sample = _get(metrics_oof, spec["null_sample"])
    model_null_feature = _get(metrics_oof, spec["null_feature"])
    model_null_mean = _get(metrics_oof, spec["null_mean"])
    pic_inter = _get(picrust_metrics, spec["intersection"])
    pic_raw = _get(picrust_metrics, spec["union_raw"])
    pic_strict = _get(picrust_metrics, spec["union_strict"])
    pic_null_sample = _get(picrust_metrics, spec["null_sample"])
    pic_null_feature = _get(picrust_metrics, spec["null_feature"])
    pic_null_mean = _get(picrust_metrics, spec["null_mean"])
    rows.append({
        "metric": name,
        "model_intersection": model_inter,
        "model_union_raw": model_raw,
        "model_union_strict": model_strict,
        "model_null_sample_mean": model_null_sample,
        "model_null_feature_mean": model_null_feature,
        "model_null_mean": model_null_mean,
        "picrust_intersection": pic_inter,
        "picrust_union_raw": pic_raw,
        "picrust_union_strict": pic_strict,
        "picrust_null_sample_mean": pic_null_sample,
        "picrust_null_feature_mean": pic_null_feature,
        "picrust_null_mean": pic_null_mean,
        "delta_intersection": (None if model_inter is None or pic_inter is None else model_inter - pic_inter),
        "delta_union_raw": (None if model_raw is None or pic_raw is None else model_raw - pic_raw),
        "delta_union_strict": (None if model_strict is None or pic_strict is None else model_strict - pic_strict),
    })

compare_table = pd.DataFrame(rows)
#save the compare_table to tsv:
compare_table.to_csv(results_dir / 'comparison_table_all_metrics.tsv', sep='	', index=False)
compare_table


Metric definitions (higher is better unless noted):

  - dm_spearman: Spearman correlation between upper triangles of Aitchison distance matrices
    (CLR space). Measures how well sample–sample relationships are preserved.

  - bray_spearman: Spearman correlation between upper triangles of Bray–Curtis distance matrices
    (TSS space). Also measures sample–sample structure, but in Bray–Curtis space.

  - procrustes_aitchison_strict: Procrustes similarity between ordinations of Aitchison distance matrices
    (CLR space). Higher means closer geometric alignment of the two ordinations.

  - procrustes_bray_strict: Procrustes similarity between ordinations of Bray–Curtis distance matrices
    (TSS space). Higher means closer geometric alignment.

  - soft_precision / soft_recall / soft_f1: Thresholded precision/recall/F1 on KO presence,
    computed in TSS space using the chosen threshold and weighting scheme.
    - soft_precision = TP / (TP + FP), meaning of all predicted present KOs, how many are truly present.
    - soft_recall = TP / (TP + FN), meaning of all truly present KOs, how many are predicted present.
    - soft_f1 = 2 * (soft_precision * soft_recall) / (soft_precision + soft_recall) (harmonic mean of precision and recall).

  - wclr_mse: Weighted MSE in CLR space (lower is better). Weights emphasize more stable features.

  - jsd: Jensen–Shannon divergence between TSS profiles (lower is better).

  - pathway_rmse: RMSE after aggregating KOs to pathways/superclasses (lower is better).

  - model_null_sample_mean = null baseline from shuffling samples (breaks sample alignment, preserves per‑sample feature distribution).
  - model_null_feature_mean = null baseline from shuffling features (breaks feature alignment, preserves per‑feature sample distribution).
  - model_null_mean = null baseline from a mean profile + noise (global average baseline).

In [ ]:
# Visualize deltas (positive = model better than PICRUSt2 for correlation metrics)
#multiply all except wclr_mse, pathway_rmse and jsd by 100 (for percentage):
compare_table_for_plotting = compare_table.copy()
#set index to metric:
compare_table_for_plotting = compare_table_for_plotting.set_index('metric')
#multiply each row except wclr_mse by 100:

for idx in compare_table_for_plotting.index:
    if idx != 'wclr_mse' and idx != 'pathway_rmse' and idx != 'jsd':
        compare_table_for_plotting.loc[idx] = compare_table_for_plotting.loc[idx]* 100

compare_table_for_plotting
        


In [ ]:
#load some colors for plotting
hex_colors = ['#240E31FF', '#CB6BCEFF', '#468892FF', '#74F3D3FF',
              '#751C6DFF', '#FDC067FF', '#AC9ECEFF', '#6EC5ABFF']


In [ ]:
results_dir

now we'll compare correspondence between kNN predictability and PICRUSt2 vs. kNN improvement over PICRUSt2

In [ ]:
#make a bar plot for dm_spearman for model_intersection, model_union_raw, model_union_strict, model_null_mean, same for picrust:
import matplotlib.pyplot as plt
labels = ['Null', 'Intersection', 'Union_raw', 'Union_strict']
metrics = ['dm_spearman', 'sample_spearman_clr', 'sample_spearman_tss', 'soft_precision', 'soft_recall', 'soft_f1', 'wclr_mse', 'jsd', 'pathway_rmse']
for metric in metrics:
    model_values = [
        compare_table_for_plotting.loc[metric, 'model_null_mean'],
        compare_table_for_plotting.loc[metric, 'model_intersection'],
        compare_table_for_plotting.loc[metric, 'model_union_raw'],
        compare_table_for_plotting.loc[metric, 'model_union_strict'],
    ]
    # picrust_values = [
    #     compare_table_for_plotting.loc[metric, 'picrust_null_mean'],
    #     compare_table_for_plotting.loc[metric, 'picrust_intersection'],
    #     compare_table_for_plotting.loc[metric, 'picrust_union_raw'],
    #     compare_table_for_plotting.loc[metric, 'picrust_union_strict'],
    # ]
    x = range(len(labels))
    width = 0.35  # the width of the bars

    fig, ax = plt.subplots()
    #set figure size:
    fig.set_size_inches(7, 3.5)
    rects1 = ax.bar([p - width/2 for p in x], model_values, width, label='kNN-model', color=hex_colors[4], alpha=0.8)
    #rects2 = ax.bar([p + width/2 for p in x], picrust_values, width, label='PICRUSt2', color=hex_colors[5], alpha=0.8)
    # Add some text for labels, title and custom x-axis tick labels, etc.
    ax.set_ylabel(f"{metric}", fontsize=14)
    ax.set_title(f'Comparison of kNN-model vs PICRUSt2 {metric} sourdough', fontsize=16, pad=15)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=14, rotation=45, ha='right')
    #set y tick labels also to fontsize 14:
    ax.tick_params(axis='y', labelsize=14)
    ax.legend(fontsize=14, frameon=True, bbox_to_anchor=(1.05, 1), loc='upper left')
    #add the values on top of the bars:
    def autolabel(rects):
        """Attach a text label above each bar in *rects*, displaying its height."""
        for rect in rects:
            height = rect.get_height()
            if height > 10:
                ax.annotate(f'{height:.1f}',
                            xy=(rect.get_x() + rect.get_width() / 2, height),
                            xytext=(0, 3),  # 3 points vertical offset
                            textcoords="offset points",
                            ha='center', va='bottom', fontsize=12)
            else:
                ax.annotate(f'{height:.2f}',
                            xy=(rect.get_x() + rect.get_width() / 2, height),
                            xytext=(0, 3),  # 3 points vertical offset
                            textcoords="offset points",
                            ha='center', va='bottom', fontsize=12)
  

    autolabel(rects1)
    #autolabel(rects2)
    #despine:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    fig.tight_layout()
    #save the figure as pdf:
    figure_path = results_dir / f'comparison_kNN_picrust2_{metric}.pdf'
    plt.savefig(figure_path, format='pdf', bbox_inches='tight', dpi=300)
    plt.show()

### Next: Sample-wise Spearman correlations (truth vs prediction) with null baselines
This shows per-sample correlations across KOs in TSS and CLR space, plus null baselines from shuffled samples and shuffled features.


In [ ]:
from intelligrate.extrapolate.metrics import (
      _pairwise_union_mats_tss,
      _pairwise_intersection_mats_tss,
      samplewise_spearman,
  )
from intelligrate.extrapolate.transforms import clr_rows
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def _sample_spearman_dist(truth_tpm, pred_tss, *, pseudocount, detect_threshold, mode="union_strict"):
      if mode == "intersection":
          truth_tss, pred_tss = _pairwise_intersection_mats_tss(truth_tpm, pred_tss, detect_threshold=0.0)
      elif mode == "union_raw":
          truth_tss, pred_tss = _pairwise_union_mats_tss(truth_tpm, pred_tss, detect_threshold=0.0, fillna_zero=True)
      else:  # union_strict
          truth_tss, pred_tss = _pairwise_union_mats_tss(truth_tpm, pred_tss, detect_threshold=detect_threshold, fillna_zero=True)

      truth_clr = clr_rows(truth_tss, pseudocount=pseudocount)
      pred_clr = clr_rows(pred_tss, pseudocount=pseudocount)
      good = truth_clr.notna().all(axis=1) & pred_clr.notna().all(axis=1)
      truth_tss, pred_tss = truth_tss.loc[good], pred_tss.loc[good]
      truth_clr, pred_clr = truth_clr.loc[good], pred_clr.loc[good]

      tss = samplewise_spearman(truth_tss, pred_tss)
      clr = samplewise_spearman(truth_clr, pred_clr)
      return tss, clr

def _null_sample_spearman_dist(truth_tpm, pred_tss, *, pseudocount, detect_threshold, n=50, seed=0):
      rng = np.random.default_rng(seed)
      truth_tss, pred_tss = _pairwise_union_mats_tss(truth_tpm, pred_tss, detect_threshold=detect_threshold, fillna_zero=True)
      truth_clr = clr_rows(truth_tss, pseudocount=pseudocount)
      pred_clr = clr_rows(pred_tss, pseudocount=pseudocount)
      good = truth_clr.notna().all(axis=1) & pred_clr.notna().all(axis=1)
      truth_tss, pred_tss = truth_tss.loc[good], pred_tss.loc[good]
      truth_clr, pred_clr = truth_clr.loc[good], pred_clr.loc[good]

      pred_tss_arr = pred_tss.to_numpy(float)
      pred_clr_arr = pred_clr.to_numpy(float)
      tss_vals, clr_vals = [], []
      for _ in range(int(n)):
          perm = rng.permutation(pred_tss_arr.shape[0])
          pred_tss_perm = pd.DataFrame(pred_tss_arr[perm, :], index=truth_tss.index, columns=truth_tss.columns)
          pred_clr_perm = pd.DataFrame(pred_clr_arr[perm, :], index=truth_clr.index, columns=truth_clr.columns)
          tss_vals.append(samplewise_spearman(truth_tss, pred_tss_perm))
          clr_vals.append(samplewise_spearman(truth_clr, pred_clr_perm))
      return pd.concat(tss_vals, axis=0), pd.concat(clr_vals, axis=0)

def _null_feature_spearman_dist(truth_tpm, pred_tss, *, pseudocount, detect_threshold, n=50, seed=1):
      rng = np.random.default_rng(seed)
      truth_tss, pred_tss = _pairwise_union_mats_tss(truth_tpm, pred_tss, detect_threshold=detect_threshold, fillna_zero=True)
      truth_clr = clr_rows(truth_tss, pseudocount=pseudocount)
      pred_clr = clr_rows(pred_tss, pseudocount=pseudocount)
      good = truth_clr.notna().all(axis=1) & pred_clr.notna().all(axis=1)
      truth_tss, pred_tss = truth_tss.loc[good], pred_tss.loc[good]
      truth_clr, pred_clr = truth_clr.loc[good], pred_clr.loc[good]

      pred_tss_arr = pred_tss.to_numpy(float)
      pred_clr_arr = pred_clr.to_numpy(float)
      tss_vals, clr_vals = [], []
      for _ in range(int(n)):
          perm = rng.permutation(pred_tss_arr.shape[1])
          pred_tss_feat = pd.DataFrame(pred_tss_arr[:, perm], index=truth_tss.index, columns=truth_tss.columns)
          pred_clr_feat = pd.DataFrame(pred_clr_arr[:, perm], index=truth_clr.index, columns=truth_clr.columns)
          tss_vals.append(samplewise_spearman(truth_tss, pred_tss_feat))
          clr_vals.append(samplewise_spearman(truth_clr, pred_clr_feat))
      return pd.concat(tss_vals, axis=0), pd.concat(clr_vals, axis=0)

def _null_mean_spearman_dist(truth_tpm, pred_tss, *, pseudocount, detect_threshold, n=50, seed=2, noise=0.01):
      rng = np.random.default_rng(seed)
      truth_tss, pred_tss = _pairwise_union_mats_tss(truth_tpm, pred_tss, detect_threshold=detect_threshold, fillna_zero=True)
      truth_clr = clr_rows(truth_tss, pseudocount=pseudocount)
      pred_clr = clr_rows(pred_tss, pseudocount=pseudocount)
      good = truth_clr.notna().all(axis=1) & pred_clr.notna().all(axis=1)
      truth_tss, pred_tss = truth_tss.loc[good], pred_tss.loc[good]
      truth_clr, pred_clr = truth_clr.loc[good], pred_clr.loc[good]

      mu_tss = truth_tss.mean(axis=0).to_numpy(float)
      mu_clr = truth_clr.mean(axis=0).to_numpy(float)
      tss_vals, clr_vals = [], []
      for _ in range(int(n)):
          noise_tss = rng.normal(0.0, noise, size=pred_tss.shape)
          noise_clr = rng.normal(0.0, noise, size=pred_clr.shape)
          pred_tss_mean = pd.DataFrame(np.clip(mu_tss[None, :] + noise_tss, 0.0, None), index=truth_tss.index, columns=truth_tss.columns)
          pred_clr_mean = pd.DataFrame(mu_clr[None, :] + noise_clr, index=truth_clr.index, columns=truth_clr.columns)
          tss_vals.append(samplewise_spearman(truth_tss, pred_tss_mean))
          clr_vals.append(samplewise_spearman(truth_clr, pred_clr_mean))
      return pd.concat(tss_vals, axis=0), pd.concat(clr_vals, axis=0)

  # --- Compute distributions for strict / raw / intersection ---
tss_model, clr_model = _sample_spearman_dist(Y, oof_fixed_tss_aligned, pseudocount=model_cfg["pseudocount_y"], detect_threshold=y_detect_threshold,
  mode="union_strict")
tss_model_raw, clr_model_raw = _sample_spearman_dist(Y, oof_fixed_tss_aligned, pseudocount=model_cfg["pseudocount_y"], detect_threshold=y_detect_threshold,
  mode="union_raw")
tss_model_int, clr_model_int = _sample_spearman_dist(Y, oof_fixed_tss_aligned, pseudocount=model_cfg["pseudocount_y"], detect_threshold=y_detect_threshold,
  mode="intersection")

tss_null, clr_null = _null_sample_spearman_dist(Y, oof_fixed_tss_aligned, pseudocount=model_cfg["pseudocount_y"], detect_threshold=y_detect_threshold, n=50,
  seed=0)
tss_feat_null, clr_feat_null = _null_feature_spearman_dist(Y, oof_fixed_tss_aligned, pseudocount=model_cfg["pseudocount_y"], detect_threshold=y_detect_threshold,
  n=50, seed=0)
tss_mean_null, clr_mean_null = _null_mean_spearman_dist(Y, oof_fixed_tss_aligned, pseudocount=model_cfg["pseudocount_y"], detect_threshold=y_detect_threshold,
  n=50, seed=0, noise=0.0001)

tss_pic = clr_pic = None
if picrust_metrics is not None:
      tss_pic, clr_pic = _sample_spearman_dist(Y, picrust, pseudocount=model_cfg["pseudocount_y"], detect_threshold=y_detect_threshold, mode="union_strict")
      #also for raw and intersection:
      tss_pic_raw, clr_pic_raw = _sample_spearman_dist(Y, picrust, pseudocount=model_cfg["pseudocount_y"], detect_threshold=y_detect_threshold, mode="union_raw")
      tss_pic_int, clr_pic_int = _sample_spearman_dist(Y, picrust, pseudocount=model_cfg["pseudocount_y"], detect_threshold=y_detect_threshold, mode="intersection")

      #and also nulls for picrust:
      tss_pic_null, clr_pic_null = _null_mean_spearman_dist(Y, picrust, pseudocount=model_cfg["pseudocount_y"], detect_threshold=y_detect_threshold, n=50,
        seed=0, noise=0.0001)
      



In [ ]:
#now, we aggregate all the metrics to plot the distributions:

tss_map = {
      "kNN_strict": tss_model,
      "kNN_raw": tss_model_raw,
      "kNN_intersection": tss_model_int,
      # "null_sample": tss_null,
      # "null_feature": tss_feat_null,
      "kNN_null_mean": tss_mean_null,
  }
if tss_pic is not None:
      tss_map["picrust2_strict"] = tss_pic
      tss_map['picrust2_raw'] = tss_pic_raw
      tss_map['picrust2_intersection'] = tss_pic_int
      tss_map['picrust2_null_mean'] = tss_pic_null

clr_map = {
      "kNN_strict": clr_model,
      "kNN_raw": clr_model_raw,
      "kNN_intersection": clr_model_int,
      # "null_sample": clr_null,
      # "null_feature": clr_feat_null,
      "kNN_null_mean": clr_mean_null,
  }
if clr_pic is not None:
      clr_map["picrust2_strict"] = clr_pic
      clr_map['picrust2_raw'] = clr_pic_raw
      clr_map['picrust2_intersection'] = clr_pic_int
      clr_map['picrust2_null_mean'] = clr_pic_null

#rearrange the order to plot:
map_order = [
      "kNN_null_mean",
      "picrust2_null_mean",
      "kNN_intersection",
      "picrust2_intersection",
      "kNN_raw",
      "picrust2_raw",
      "kNN_strict",
      "picrust2_strict",
  ]  
tss_map = {k: tss_map[k] for k in map_order if k in tss_map}
clr_map = {k: clr_map[k] for k in map_order if k in clr_map}          


define all the plotting utilities:

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests
from itertools import combinations

def pairwise_wilcoxon_cld(data_map, alpha=0.05, zero_method="wilcox"):
      """
      Paired Wilcoxon signed-rank with BH correction + CLD.
      data_map: dict[label -> pd.Series with SAME index].
      """
      labels = list(data_map.keys())
      # align indices
      common = None
      for k, v in data_map.items():
          if common is None:
              common = v.index
          else:
              common = common.intersection(v.index)
      aligned = {k: data_map[k].loc[common].astype(float) for k in labels}

      rows = []
      for a, b in combinations(labels, 2):
          x, y = aligned[a], aligned[b]
          # drop NaNs pairwise
          m = np.isfinite(x) & np.isfinite(y)
          if m.sum() < 10:
              p = np.nan
          else:
              p = wilcoxon(x[m], y[m], zero_method=zero_method).pvalue
          rows.append((a, b, p))
      p_df = pd.DataFrame(rows, columns=["A", "B", "pval"])

      # FDR correction
      pvals = p_df["pval"].to_numpy()
      mask = np.isfinite(pvals)
      adj = np.full_like(pvals, np.nan, dtype=float)
      if mask.sum() > 0:
          adj[mask] = multipletests(pvals[mask], alpha=alpha, method="fdr_bh")[1]
      p_df["p_adj"] = adj

      # significance matrix
      sig = {a: {b: False for b in labels} for a in labels}
      for _, row in p_df.iterrows():
          if np.isfinite(row["p_adj"]) and row["p_adj"] < alpha:
              sig[row["A"]][row["B"]] = True
              sig[row["B"]][row["A"]] = True

      # greedy CLD
      cld = {k: "" for k in labels}
      letters = []
      for label in labels:
          placed = False
          for letter in letters:
              conflict = False
              for other, ltr in cld.items():
                  if ltr == letter and sig[label][other]:
                      conflict = True
                      break
              if not conflict:
                  cld[label] = letter
                  placed = True
                  break
          if not placed:
              new_letter = chr(ord("a") + len(letters))
              letters.append(new_letter)
              cld[label] = new_letter

      return cld, p_df

def _ensure_sample_indexed(series_or_array, sample_index):
      """
      Make sure each group is a Series indexed by samples.
      If the input is a concatenated null (length = n_samples * R),
      compress it to per-sample mean.
      """
      v = series_or_array
      if not isinstance(v, pd.Series):
          v = pd.Series(v)

      n = len(sample_index)
      if len(v) == n:
          return v.copy().set_axis(sample_index)

      # Assume concatenated replicates (n * R)
      if len(v) % n == 0:
          R = len(v) // n
          arr = v.to_numpy().reshape(R, n).T  # (n, R)
          return pd.Series(np.nanmean(arr, axis=1), index=sample_index)

      raise ValueError(f"Length {len(v)} is not compatible with n_samples={n}")

sample_index = tss_model.index  # any per-sample Series index
tss_map_aligned = {}
for k, v in tss_map.items():
      if v is None:
          continue
      tss_map_aligned[k] = _ensure_sample_indexed(v, sample_index)

clr_map_aligned = {}
for k, v in clr_map.items():
      if v is None:
          continue
      clr_map_aligned[k] = _ensure_sample_indexed(v, sample_index)

def _plot_violin_with_cld_paired(
      data_map,
      title,
      alpha=0.05,
      colors=None,   # dict label -> color
      dot_alpha=0.5,
      letter_offset=0.02,
  ):
      labels = list(data_map.keys())
      data = [data_map[k].dropna().to_numpy(float) for k in labels]

      fig, ax = plt.subplots(figsize=(5, 4.5))
      parts = ax.violinplot(data, showmeans=True, showextrema=False)

      for i, body in enumerate(parts['bodies']):
                label = labels[i]
                c = colors.get(label, "#999999") if colors else "#999999"
                body.set_facecolor(c)
                body.set_edgecolor("none")
                body.set_alpha(0.3)

      # Set color black for mean lines:
      if 'cmeans' in parts:
          parts['cmeans'].set_color('black')
          parts['cmeans'].set_linewidth(1.5)

        # Dots with matching color
      for i, vals in enumerate(data, start=1):
            label = labels[i-1]
            c = colors.get(label, "#666666") if colors else "#666666"
            x = np.random.normal(i, 0.04, size=len(vals))
            ax.scatter(x, vals, s=8, alpha=dot_alpha, color=c)

      ax.set_xticks(range(1, len(labels) + 1))
      ax.set_xticklabels(labels, rotation=45, fontsize=14, ha='right')
      ax.tick_params(axis='y', labelsize=14)
      ax.set_ylabel("sample-wise Spearman", fontsize=14)
      ax.set_ylim(0.0, 1.1)
      ax.set_title(title, pad=15, fontsize=16)

      cld, p_df = pairwise_wilcoxon_cld(data_map, alpha=alpha)

      # Place CLD letters right above each violin
      for i, lab in enumerate(labels, start=1):
          vals = data[i-1]
          if len(vals) == 0:
              continue
          y = np.nanmax(vals) + letter_offset
          ax.text(i, y, cld.get(lab, ""), ha="center", va="bottom", fontsize=14)
      #despine:
      ax.spines['top'].set_visible(False)
      ax.spines['right'].set_visible(False)  
      fig.tight_layout()
      #save as pdf, with title in name:
      figure_path = results_dir / f'violin_{title.replace(" ", "_").lower()}.pdf'
      plt.savefig(figure_path, format='pdf', bbox_inches='tight', dpi=300)
      plt.show()
      return cld, p_df

  #Example color mapping (distinct model vs picrust):

colors = {
      "kNN_strict": hex_colors[4],
      "kNN_raw": hex_colors[4],
      "kNN_intersection": hex_colors[4],
      "picrust2_strict": hex_colors[5],
      "picrust2_raw": hex_colors[5],
      "picrust2_intersection": hex_colors[5],
      "kNN_null_mean": hex_colors[4],
      "picrust2_null_mean": hex_colors[5],
  }

cld_tss, p_tss = _plot_violin_with_cld_paired(tss_map_aligned, "TSS sample-wise Spearman sourdough",
  colors=colors)
cld_clr, p_clr = _plot_violin_with_cld_paired(clr_map_aligned, "CLR sample-wise Spearman sourdough",
  colors=colors)

## Pathway deviation plots (log10)
We compare pathway-level deviations vs. truth for kNN-model and PICRUSt2 (raw vs union and intersection).

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon

def _aggregate_to_pathway(tss, ko_to_group):
    common = [c for c in tss.columns if c in ko_to_group]
    if not common:
        return pd.DataFrame(index=tss.index)
    groups = [ko_to_group[c] for c in common]
    out = tss.loc[:, common].copy()
    out.columns = groups
    return out.groupby(level=0, axis=1).sum()

def _align_union(a, b):
    cols = a.columns.union(b.columns)
    return a.reindex(columns=cols, fill_value=0.0), b.reindex(columns=cols, fill_value=0.0)

def _align_union_strict(a, b, detect_threshold=1e-6):
    cols = a.columns.union(b.columns)
    a_re = a.reindex(columns=cols, fill_value=0.0)
    b_re = b.reindex(columns=cols, fill_value=0.0)
    a_re = a_re.where(a_re >= detect_threshold, 0.0)
    b_re = b_re.where(b_re >= detect_threshold, 0.0)
    return a_re, b_re

def _align_intersection(a, b):
    cols = a.columns.intersection(b.columns)
    return a.loc[:, cols], b.loc[:, cols]

# Build per-pathway deviations for ALL configs (kNN + PICRUSt2 + nulls + intersection/raw/strict)
def make_all_pathway_devs(
    truth_pw, model_pw, pic_pw=None,
    detect_threshold=1e-6,
    n_null=50, seed=0, noise=0.0001
):
    rng = np.random.default_rng(seed)

    def build_dev(ref, df, cfg):
        common = df.index.intersection(ref.index)
        A = df.loc[common]
        R = ref.loc[common]
        dev = np.log10(A + 1e-6) - np.log10(R + 1e-6)
        long = dev.stack().reset_index()
        long.columns = ['sample', 'pathway', 'deviation_log10']
        long['cfg'] = cfg
        return long

    rows = []

    # intersection
    _, m_int = _align_intersection(truth_pw, model_pw)
    rows.append(build_dev(truth_pw, m_int, 'kNN_intersection'))
    if pic_pw is not None:
        _, p_int = _align_intersection(truth_pw, pic_pw)
        rows.append(build_dev(truth_pw, p_int, 'picrust2_intersection'))

    # union_raw
    _, m_raw = _align_union(truth_pw, model_pw)
    rows.append(build_dev(truth_pw, m_raw, 'kNN_raw'))
    if pic_pw is not None:
        _, p_raw = _align_union(truth_pw, pic_pw)
        rows.append(build_dev(truth_pw, p_raw, 'picrust2_raw'))

    # union_strict
    _, m_strict = _align_union_strict(truth_pw, model_pw, detect_threshold=detect_threshold)
    rows.append(build_dev(truth_pw, m_strict, 'kNN_strict'))
    if pic_pw is not None:
        _, p_strict = _align_union_strict(truth_pw, pic_pw, detect_threshold=detect_threshold)
        rows.append(build_dev(truth_pw, p_strict, 'picrust2_strict'))

    # nulls (union_strict baseline)
    _, m_base = _align_union_strict(truth_pw, model_pw, detect_threshold=detect_threshold)
    m_arr = m_base.to_numpy(float)
    for _ in range(int(n_null)):
        perm = rng.permutation(m_arr.shape[0])
        m_s = pd.DataFrame(m_arr[perm, :], index=truth_pw.index, columns=m_base.columns)
        rows.append(build_dev(truth_pw, m_s, 'kNN_null_sample'))

        permc = rng.permutation(m_arr.shape[1])
        m_f = pd.DataFrame(m_arr[:, permc], index=truth_pw.index, columns=m_base.columns)
        rows.append(build_dev(truth_pw, m_f, 'kNN_null_feature'))

        mu = truth_pw.mean(axis=0).to_numpy(float)
        noise_t = rng.normal(0, noise, size=m_arr.shape)
        m_m = pd.DataFrame(np.clip(mu[None, :] + noise_t, 0, None), index=truth_pw.index, columns=m_base.columns)
        rows.append(build_dev(truth_pw, m_m, 'kNN_null_mean'))

    if pic_pw is not None:
        _, p_base = _align_union_strict(truth_pw, pic_pw, detect_threshold=detect_threshold)
        p_arr = p_base.to_numpy(float)
        for _ in range(int(n_null)):
            perm = rng.permutation(p_arr.shape[0])
            p_s = pd.DataFrame(p_arr[perm, :], index=truth_pw.index, columns=p_base.columns)
            rows.append(build_dev(truth_pw, p_s, 'picrust2_null_sample'))

            permc = rng.permutation(p_arr.shape[1])
            p_f = pd.DataFrame(p_arr[:, permc], index=truth_pw.index, columns=p_base.columns)
            rows.append(build_dev(truth_pw, p_f, 'picrust2_null_feature'))

            mu = truth_pw.mean(axis=0).to_numpy(float)
            noise_t = rng.normal(0, noise, size=p_arr.shape)
            p_m = pd.DataFrame(np.clip(mu[None, :] + noise_t, 0, None), index=truth_pw.index, columns=p_base.columns)
            rows.append(build_dev(truth_pw, p_m, 'picrust2_null_mean'))

    return pd.concat(rows, ignore_index=True)

# --- Aggregate to pathways ---
truth_pw = _aggregate_to_pathway(Y.div(Y.sum(axis=1).replace(0, np.nan), axis=0), ko_to_superclass)
model_pw = _aggregate_to_pathway(oof_fixed_tss_aligned, ko_to_superclass)

pic_pw = None
if 'picrust' in locals() and picrust is not None:
    pic_pw = _aggregate_to_pathway(picrust, ko_to_superclass)
    pic_pw = pic_pw.div(pic_pw.sum(axis=1).replace(0, np.nan), axis=0)

# Build one combined dev table with all configs
all_dev = make_all_pathway_devs(
    truth_pw, model_pw, pic_pw,
    detect_threshold=model_cfg['y_detect_threshold'],
    n_null=50,
    seed=0,
    noise=0.0001,
)

# Collapse repeated null draws to one value per sample/pathway (mean across null iterations)
all_dev = (
    all_dev
    .groupby(["cfg", "sample", "pathway"], as_index=False)["deviation_log10"]
    .mean()
)



# Example: plot all configs for each pathway
cfg_order = [
    'kNN_null_mean', 'picrust2_null_mean',
    # 'kNN_null_sample', 'picrust2_null_sample',
    # 'kNN_null_feature', 'picrust2_null_feature',
    'kNN_intersection', 'picrust2_intersection',
    'kNN_raw', 'picrust2_raw',
    'kNN_strict', 'picrust2_strict',
]


In [ ]:
all_dev

define all the plotting utilities for pathway deviation plots:

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
import seaborn as sns
import matplotlib.pyplot as plt

def bh_fdr(pvals: np.ndarray) -> np.ndarray:
    p = np.asarray(pvals, float)
    out = np.full_like(p, np.nan)
    ok = np.isfinite(p)
    if ok.sum() == 0:
        return out
    pv = p[ok]
    order = np.argsort(pv)
    ranked = pv[order]
    m = len(ranked)
    q = ranked * m / (np.arange(1, m + 1))
    q = np.minimum.accumulate(q[::-1])[::-1]
    tmp = np.empty_like(pv)
    tmp[order] = np.clip(q, 0, 1)
    out[ok] = tmp
    return out

def pairwise_paired_wilcoxon_for_pathway(dev_long: pd.DataFrame, pathway: str, cfg_order=None,
                                         min_pairs=10, zero_method='wilcox',
                                         alternative='two-sided') -> pd.DataFrame:
    d = dev_long[dev_long['pathway'] == pathway].copy()
    if d.empty:
        return pd.DataFrame(columns=['pathway','cfg1','cfg2','p','p_adj','n_pairs'])
    if cfg_order is None:
        cfgs = sorted(d['cfg'].unique())
    else:
        cfgs = [c for c in cfg_order if c in d['cfg'].unique()]
    by = {cfg: d[d['cfg'] == cfg].set_index('sample')['deviation_log10'] for cfg in cfgs}
    rows = []
    for i in range(len(cfgs)):
        for j in range(i + 1, len(cfgs)):
            a, b = cfgs[i], cfgs[j]
            sa, sb = by[a], by[b]
            common = sa.index.intersection(sb.index)
            if len(common) < min_pairs:
                rows.append({'pathway': pathway, 'cfg1': a, 'cfg2': b, 'p': np.nan, 'n_pairs': len(common)})
                continue
            diff = (sa.loc[common] - sb.loc[common]).to_numpy(dtype=float)
            diff = diff[np.isfinite(diff)]
            if diff.size < min_pairs:
                rows.append({'pathway': pathway, 'cfg1': a, 'cfg2': b, 'p': np.nan, 'n_pairs': int(diff.size)})
                continue
            if np.allclose(diff, 0):
                p = 1.0
            else:
                p = float(wilcoxon(diff, zero_method=zero_method, alternative=alternative).pvalue)
            rows.append({'pathway': pathway, 'cfg1': a, 'cfg2': b, 'p': p, 'n_pairs': int(diff.size)})
    out = pd.DataFrame(rows)
    out['p_adj'] = bh_fdr(out['p'].to_numpy())
    return out

def wilcoxon_vs_zero_for_pathway(dev_long: pd.DataFrame, pathway: str, cfg_order=None,
                                 min_n=10, zero_method='wilcox',
                                 alternative='two-sided') -> pd.DataFrame:
    d = dev_long[dev_long['pathway'] == pathway].copy()
    if d.empty:
        return pd.DataFrame(columns=['pathway','cfg','p','p_adj','n','median'])
    if cfg_order is None:
        cfgs = sorted(d['cfg'].unique())
    else:
        cfgs = [c for c in cfg_order if c in d['cfg'].unique()]
    rows = []
    for cfg in cfgs:
        x = d.loc[d['cfg'] == cfg, 'deviation_log10'].to_numpy(dtype=float)
        x = x[np.isfinite(x)]
        n = int(x.size)
        if n < min_n:
            rows.append({'pathway': pathway, 'cfg': cfg, 'p': np.nan, 'n': n, 'median': np.nan})
            continue
        med = float(np.median(x))
        if np.allclose(x, 0):
            p = 1.0
        else:
            p = float(wilcoxon(x, zero_method=zero_method, alternative=alternative).pvalue)
        rows.append({'pathway': pathway, 'cfg': cfg, 'p': p, 'n': n, 'median': med})
    out = pd.DataFrame(rows)
    out['p_adj'] = bh_fdr(out['p'].to_numpy())
    return out

def compute_cld_letters(cfgs, pairwise_df, p_col='p_adj', alpha=0.05, value_by_cfg=None):
    p_lookup = {}
    for _, r in pairwise_df.iterrows():
        a, b = r['cfg1'], r['cfg2']
        p_lookup[tuple(sorted((a, b)))] = r[p_col]
    def nonsig(a, b):
        if a == b:
            return True
        p = p_lookup.get(tuple(sorted((a, b))), np.nan)
        if not np.isfinite(p):
            return False
        return p >= alpha
    if value_by_cfg is not None:
        order = list(value_by_cfg.loc[cfgs].sort_values(ascending=False).index)
    else:
        order = list(cfgs)
    letter_groups = []
    for cfg in order:
        placed = False
        for grp in letter_groups:
            if all(nonsig(cfg, other) for other in grp):
                grp.append(cfg)
                placed = True
                break
        if not placed:
            letter_groups.append([cfg])
    def idx_to_letters(i):
        s = ''
        i0 = i
        while True:
            s = chr(ord('a') + (i0 % 26)) + s
            i0 = i0 // 26 - 1
            if i0 < 0:
                break
        return s
    cld = {cfg: '' for cfg in cfgs}
    for gi, grp in enumerate(letter_groups):
        L = idx_to_letters(gi)
        for cfg in grp:
            cld[cfg] += L
    return cld


def _violin_points_mpl(
    d: pd.DataFrame,
    cfgs,
    ax,
    colors=None,         # dict cfg -> color
    y_col="deviation_log10",
    dot_alpha=0.5,
    dot_size=15,
    violin_alpha=0.30,
    jitter_sd=0.04,
    showmeans=True,
):
    """
    Matplotlib violins styled like your second snippet:
    - per-group colors
    - semi-transparent violins
    - black mean line
    - dots colored by group
    """
    data = [d.loc[d["cfg"] == cfg, y_col].to_numpy(float) for cfg in cfgs]

    parts = ax.violinplot(
        data,
        showmeans=showmeans,
        showextrema=False,
        widths=0.85,
    )

    # color violins
    for i, body in enumerate(parts["bodies"]):
        cfg = cfgs[i]
        c = (colors or {}).get(cfg, "#999999")
        body.set_facecolor(c)
        body.set_edgecolor("none")
        body.set_alpha(violin_alpha)

    # black mean line
    if showmeans and "cmeans" in parts:
        parts["cmeans"].set_color("black")
        parts["cmeans"].set_linewidth(1.5)

    # dots colored by cfg
    for i, cfg in enumerate(cfgs, start=1):
        vals = d.loc[d["cfg"] == cfg, y_col].to_numpy(float)
        vals = vals[np.isfinite(vals)]
        if vals.size == 0:
            continue
        c = (colors or {}).get(cfg, "#666666")
        x = np.random.normal(i, jitter_sd, size=vals.size)
        ax.scatter(x, vals, s=dot_size, alpha=dot_alpha, color=c, linewidths=0)

    ax.set_xticks(range(1, len(cfgs) + 1))
    ax.set_xticklabels(cfgs, rotation=45, ha="right")


def _apply_axis_style(ax, title, ylabel):
    ax.axhline(0, lw=1, color="black")
    ax.set_title(title, fontsize=16, loc="left", pad=15)
    ax.set_xlabel("")
    ax.set_ylabel(ylabel, fontsize=14)
    ax.tick_params(axis="x", labelsize=14)
    ax.tick_params(axis="y", labelsize=14)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


# --- Replace ONLY the plotting parts in your existing functions ---

def plot_pathway_pairwise_cld(
    dev_long: pd.DataFrame,
    pathway: str,
    pairwise_df: pd.DataFrame,
    cfg_order=None,
    kind="violin",              # keep "box" as an option if you want
    palette=None,               # pass dict cfg -> color here
    figsize=(5, 5),
    sig_cutoff=0.05,
    p_col="p_adj",
    cld_order_by="median",
    y_max_cap=None,
    savepath=None,
):
    d = dev_long[dev_long["pathway"] == pathway].copy()
    if d.empty:
        raise ValueError(f"No data for pathway '{pathway}'.")

    cfgs = sorted(d["cfg"].unique()) if cfg_order is None else [c for c in cfg_order if c in d["cfg"].unique()]
    if y_max_cap is not None:
        d = d[np.abs(d["deviation_log10"]) <= y_max_cap]

    fig, ax = plt.subplots(figsize=figsize)

    if kind == "violin":
        _violin_points_mpl(
            d, cfgs, ax,
            colors=palette,            # dict cfg -> color
            y_col="deviation_log10",
            dot_alpha=0.5,
            dot_size=15,
            violin_alpha=0.30,
            jitter_sd=0.04,
            showmeans=True,
        )
    else:
        # keep your existing seaborn boxplot if desired
        import seaborn as sns
        sns.boxplot(
            data=d, x="cfg", y="deviation_log10",
            order=cfgs, palette=palette, showfliers=False, linewidth=2, ax=ax,
            boxprops=dict(alpha=0.3),
        )
        # color points per cfg (use hue to get palette mapping)
        sns.stripplot(
            data=d, x="cfg", y="deviation_log10",
            order=cfgs, hue="cfg", palette=palette,
            dodge=False, jitter=0.25, size=20, alpha=0.5, ax=ax,
        )
        ax.legend_.remove()

    _apply_axis_style(ax, f"{pathway} —\n pairwise Wilcoxon sourdough", "log10 dev. vs truth")

    # CLD letters (unchanged logic)
    if cld_order_by == "median":
        stat = d.groupby("cfg")["deviation_log10"].median()
    elif cld_order_by == "mean":
        stat = d.groupby("cfg")["deviation_log10"].mean()
    else:
        stat = None

    cld = compute_cld_letters(cfgs, pairwise_df, p_col=p_col, alpha=sig_cutoff, value_by_cfg=stat)

    group_max = d.groupby("cfg")["deviation_log10"].max()
    y_range = (d["deviation_log10"].max() - d["deviation_log10"].min()) if len(d) else 1.0
    offset = max(0.05, 0.05 * y_range)

    # x positions differ between mpl-violin (1..K) and seaborn categorical (0..K-1)
    use_mpl_positions = (kind == "violin")
    for i, cfg in enumerate(cfgs, start=(1 if use_mpl_positions else 0)):
        if cfg not in group_max or pd.isna(group_max[cfg]):
            continue
        ax.text(i, float(group_max[cfg]) + offset-0.15, cld.get(cfg, ""), ha="center", va="bottom", fontsize=14)

    cur_ymin, cur_ymax = ax.get_ylim()
    ax.set_ylim(cur_ymin, max(cur_ymax, float(group_max.max()) + 1 * offset))
    if y_max_cap is not None:
        ax.set_ylim(-y_max_cap, y_max_cap)
    #add pad to title:
    
    fig.tight_layout()
    if savepath:
        fig.savefig(savepath, dpi=300, bbox_inches="tight")
    plt.show()
    return fig
def p_to_stars(p):
    if not np.isfinite(p):
        return ''
    if p < 0.001:
        return '***'
    if p < 0.01:
        return '**'
    if p < 0.05:
        return '*'
    return 'ns'


def plot_pathway_vs_zero(
    dev_long: pd.DataFrame,
    pathway: str,
    vs0_df: pd.DataFrame,
    cfg_order=None,
    kind="violin",
    palette=None,               # dict cfg -> color
    figsize=(5, 5),
    sig_cutoff=0.05,
    p_col="p_adj",
    label_style="stars",
    y_max_cap=None,
    savepath=None,
):
    d = dev_long[dev_long["pathway"] == pathway].copy()
    if d.empty:
        raise ValueError(f"No data for pathway '{pathway}'.")

    cfgs = sorted(d["cfg"].unique()) if cfg_order is None else [c for c in cfg_order if c in d["cfg"].unique()]
    if y_max_cap is not None:
        d = d[np.abs(d["deviation_log10"]) <= y_max_cap]

    fig, ax = plt.subplots(figsize=figsize)

    if kind == "violin":
        _violin_points_mpl(d, cfgs, ax, colors=palette, y_col="deviation_log10")
    else:
        import seaborn as sns
        sns.boxplot(
            data=d, x="cfg", y="deviation_log10",
            order=cfgs, palette=palette, showfliers=False, linewidth=2, ax=ax,
            boxprops=dict(alpha=0.3),
        )
        sns.stripplot(
            data=d, x="cfg", y="deviation_log10",
            order=cfgs, hue="cfg", palette=palette,
            dodge=False, jitter=0.25, size=20, alpha=0.5, ax=ax,
        )
        ax.legend_.remove()

    _apply_axis_style(ax, f"{pathway} —\n Wilcoxon vs 0 sourdough", "log10 dev. vs truth")

    group_max = d.groupby("cfg")["deviation_log10"].max()
    y_range = (d["deviation_log10"].max() - d["deviation_log10"].min()) if len(d) else 1.0
    offset = max(0.05, 0.05 * y_range)

    vs0_df = vs0_df.set_index("cfg")
    use_mpl_positions = (kind == "violin")
    for i, cfg in enumerate(cfgs, start=(1 if use_mpl_positions else 0)):
        if cfg not in group_max or pd.isna(group_max[cfg]):
            continue
        q = float(vs0_df.loc[cfg, p_col]) if cfg in vs0_df.index else np.nan
        lab = (f"q={q:.3g}" if (label_style == "q" and np.isfinite(q)) else p_to_stars(q))
        ax.text(i, float(group_max[cfg]) + offset-0.15, lab, ha="center", va="bottom", fontsize=14)

    cur_ymin, cur_ymax = ax.get_ylim()
    ax.set_ylim(cur_ymin, max(cur_ymax, float(group_max.max()) +  1* offset))
    if y_max_cap is not None:
        ax.set_ylim(-y_max_cap, y_max_cap)

    fig.tight_layout()
    if savepath:
        fig.savefig(savepath, dpi=300, bbox_inches="tight")
    plt.show()
    return fig

colors = {
    "kNN_strict": hex_colors[4],
    "kNN_raw": hex_colors[4],
    "kNN_intersection": hex_colors[4],
    "picrust2_strict": hex_colors[5],
    "picrust2_raw": hex_colors[5],
    "picrust2_intersection": hex_colors[5],
    "kNN_null_mean": hex_colors[4],
    "picrust2_null_mean": hex_colors[5],
}
def plot_two_per_pathway(
    dev_long: pd.DataFrame,
    pathway: str,
    cfg_order=None,
    kind='box',
    palette=None,
    min_pairs=10,
    min_n=10,
    sig_cutoff=0.05,
    y_max_cap=None,
    outdir='results/pathway_stats',
):
    import os
    os.makedirs(outdir, exist_ok=True)
    pairwise_df = pairwise_paired_wilcoxon_for_pathway(
        dev_long, pathway, cfg_order=cfg_order, min_pairs=min_pairs
    )
    vs0_df = wilcoxon_vs_zero_for_pathway(
        dev_long, pathway, cfg_order=cfg_order, min_n=min_n
    )
    safe = pathway.replace('/', '_').replace(' ', '_')
    plot_pathway_pairwise_cld(
        dev_long, pathway, pairwise_df,
        cfg_order=cfg_order, kind=kind, palette=palette,
        sig_cutoff=sig_cutoff, y_max_cap=y_max_cap,
        savepath=f"{outdir}/{safe}__pairwise_cld.pdf",
     


    )
    plot_pathway_vs_zero(
        dev_long, pathway, vs0_df,
        cfg_order=cfg_order, kind=kind, palette=palette,
        sig_cutoff=sig_cutoff, y_max_cap=y_max_cap,
        label_style='stars',
        savepath=f"{outdir}/{safe}__vs0.pdf"
    )
    return pairwise_df, vs0_df

# pathway = 'Amino acid metabolism'
for pathway in all_dev['pathway'].unique():
    pairwise_df, vs0_df = plot_two_per_pathway(
        all_dev,
        pathway=pathway,
        cfg_order=cfg_order,
        kind="violin",
        palette=colors,   # <-- dict cfg->color
        outdir=str(results_dir / "pathway_stats"),

    )


remove the 'strict' metrics since those are massively skewing the visual:

In [ ]:
cfg_order = ['kNN_null_mean',
 'picrust2_null_mean',
 'kNN_intersection',
 'picrust2_intersection',
 'kNN_raw',
 'picrust2_raw',
#  'kNN_strict',
#  'picrust2_strict'
 ]

for pathway in all_dev['pathway'].unique():
    pairwise_df, vs0_df = plot_two_per_pathway(
        all_dev,
        pathway=pathway,
        cfg_order=cfg_order,
        kind="violin",
        palette=colors,   # <-- dict cfg->color
        outdir=str(results_dir / "pathway_stats2"),

    )

In [ ]:
all_dev

## Interpreting outputs
- `results/oof_clr.tsv`, `results/oof_tss.tsv`: out-of-fold predictions from nested CV
- `results/folds.tsv`: per-fold metrics + selected hyperparameters
- `results/summary.json` / `results/summary.tsv`: overall metrics
- `results/embed.joblib`: fitted embedding for k-mers
- `results/model.joblib`: final model artifact
- `results/pred_full.*`: predictions for all samples
- `results/pred_paired.metrics.tsv`: metrics when truth is provided
